In [1]:
import pandas as pd
df = pd.read_csv('C:/Users/chalani/Desktop/smartcare-ai-coursework/data/raw/smartcare_ai_dataset_1000.csv')

In [2]:
# Flag the admitted=1 but room_type missing
df['room_type_missing_flag'] = df['room_type'].isnull() & (df['admitted'] == 1)

In [3]:
# Fill the missing room_type values (906) identified in data understanding section
df['room_type'] = df['room_type'].fillna('Not Recorded')

In [4]:
# Confirm no missing values remain
df.isnull().sum().sum()

np.int64(0)

In [5]:
# Re-check duplicates
df.duplicated().sum() 

np.int64(0)

In [6]:
# Look at the low BMI rows (check with age)
df[df['bmi'] < 15][['age', 'bmi']]

,age,bmi
45,50,14.0
90,40,14.7
141,36,14.4
154,68,14.5
245,57,14.0
293,18,14.5
322,40,14.0
333,6,14.0
564,62,14.0


In [7]:
# Flag (without delete) the BMI outlier
df['bmi_outlier_flag'] = df['bmi'] < 15

In [8]:
# Admitted patients should generally have some room charge - check for admitted=1 but room_charge=0
df[(df['admitted'] == 1) & (df['room_charge_lkr'] == 0)][['admitted', 'room_charge_lkr']]

,admitted,room_charge_lkr
19,1,0
21,1,0
29,1,0
30,1,0
32,1,0
...,...,...
971,1,0
974,1,0
977,1,0
983,1,0


In [9]:
# Flag the admitted=1 but room_charge=0
df['room_charge_missing_flag'] = (df['admitted'] == 1) & (df['room_charge_lkr'] == 0)

In [13]:
# Checks whether the "missing room type" group and the "zero room charge" group are the same 236 patients or different ones
(df['room_type_missing_flag'] & df['room_charge_missing_flag']).sum()

np.int64(236)

In [10]:
# Admitted patients should have length_of_stay > 0
df[(df['admitted'] == 1) & (df['length_of_stay_days'] == 0)]

,record_id,patient_id,age,gender,blood_group,department,diagnosis,appointment_date,waiting_days,previous_appointments,...,medicine_charge_lkr,total_bill_lkr,payment_status,payment_method,no_show,readmitted_30_days,disease_risk_level,room_type_missing_flag,bmi_outlier_flag,room_charge_missing_flag


In [11]:
# Non-admitted patients shouldn't have a room charge at all
df[(df['admitted'] == 0) & (df['room_charge_lkr'] > 0)]

,record_id,patient_id,age,gender,blood_group,department,diagnosis,appointment_date,waiting_days,previous_appointments,...,medicine_charge_lkr,total_bill_lkr,payment_status,payment_method,no_show,readmitted_30_days,disease_risk_level,room_type_missing_flag,bmi_outlier_flag,room_charge_missing_flag


## Data Cleaning Notes

1. room_type: 906 missing values total. 670 are patients who weren't admitted 
   (expected, filled as "Not Recorded"). The remaining 236 are admitted patients 
   missing room_type — flagged with room_type_missing_flag rather than deleted.

2. bmi: 9 rows have bmi below 15. Checked against age — 8 of 9 are adults, so this 
   is a genuine outlier, not explained by young patients. Flagged with 
   bmi_outlier_flag rather than removed, since deleting rows without strong 
   justification isn't good practice.

3. room_charge_lkr: The same 236 admitted patients missing room_type also have 
   zero room_charge_lkr (confirmed with 
   (df['room_type_missing_flag'] & df['room_charge_missing_flag']).sum(), which 
   returned 236 — a full match). This means it's one single data gap, not two 
   separate issues — this subgroup's room-related records were never completed. 
   Flagged with room_charge_missing_flag. Not removed, since dropping 236 rows 
   would mean losing 23.6% of all admitted patients.

No duplicate rows. No issues found with length_of_stay or non-admitted billing.